In [1]:
from google import genai
from google.genai import types

In [ ]:
pip install panel

In [2]:
import panel as pn  # GUI
pn.extension()

import utils  # Ensure utils.py is available and correct

In [3]:
client = genai.Client(api_key="APIKey")

In [4]:
print([m.name for m in client.models.list()])

['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-4-26b-a4b-it', 'models/gemma-4-31b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3-pro-image-preview', 'models/gemini-3-pro-image', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-omni-flash-preview', 'models/lyria-3-clip-preview', 'models/lyria

In [5]:
chat = client.chats.create(
    model="gemini-3.1-flash-lite")
    



In [6]:
def get_moderation_for_message(message):
    # Moderation using Gemini (prompt-based approach)
    moderation_prompt = f"""
    You are a content moderator. Analyze the following text for harmful content, hate speech, harassment, or unsafe topics.
    Respond with a single letter:
    Y - if it contains harmful content
    N - if it is safe.
    
    Text: '''{message}'''
    """
    msgs = [
        {
            "role": "user",
            "content": f"{moderation_prompt}"
        }
    ]
    return utils.get_completion_from_messages(messages=msgs)

In [10]:
# Function to process user messages
def process_user_message(user_input, all_messages, debug=True):
    delimiter = "```"

    # Step 1: Input validation
    moderation_result = get_moderation_for_message(user_input)
    if debug:
        if (moderation_result == 'Y'):
            print("Step 1: Input message did not pass the modearation check. Hence Exit")
            return
        else:
            print("Step 1: Input passed moderation check.")
        
    # Step 2: Extract products and categories
    category_and_product_response = utils.find_category_and_product_only(
        user_input, utils.get_products_and_category()
    )
    
    if debug:
        print(category_and_product_response)
    category_and_product_list = utils.read_string_to_list(category_and_product_response)
    if debug:
        print(category_and_product_list)

    if debug:
        print(f"Step 2: Extracted products: {category_and_product_list}")

    # Step 3: Fetch product information
    product_information = utils.generate_output_string(category_and_product_list)

    if debug:
        print("Step 3: Retrieved product information.")

    # Step 4: Generate customer service response
    step_4_system_message_content = "You are a helpful AI assistant. Provide detailed and accurate information."

    msgs = [
        {
            "role": "user",
            "content": f"\n\n{delimiter}{user_input}{delimiter}",
        },  # ✅ Merged system message
        {"role": "system", "content": f"{step_4_system_message_content}"},
    ]

    final_response = utils.get_completion_from_messages(msgs)
    # all_messages.append({"role": "assistant", "content": final_response})

    if debug:
        print("Step 4: Generated response.")

    # Step 5: Evaluate response quality
    step_5_system_message_content = "Evaluate whether the AI response fully answers the customer's question."

    evaluation_message = f"""\n\nCustomer message: {delimiter}{user_input}{delimiter}\n
    Agent response: {delimiter}{final_response}{delimiter}\n
    Does the response sufficiently answer the question? reply in single word Y or N"""
    msgs = [
        {
            "role": "user",
            "content": f"{evaluation_message}"
        },
        {"role": "system", "content": f"{step_5_system_message_content}"},
    ]
   
    evaluation_response = utils.get_completion_from_messages(messages=msgs)

    # Ensure strict formatting
    cleaned_evaluation_response = evaluation_response.strip(). strip().upper()  # Remove spaces & enforce uppercase
    moderation_response_result = get_moderation_for_message(cleaned_evaluation_response)
    if debug:
        print(f"Step 5: Evaluation result: {cleaned_evaluation_response}")

    # Step 7: Decision based on evaluation
    if moderation_response_result == "N":
        if debug:
            print("Step 6: Response is approved.")
        return final_response #, all_messages
    else:
        if debug:
            print("Step 6: Response is did not pass moderation.")
        return "I'm unable to provide the information you're looking for. Let me connect you with a representative." #, all_messages

# Test the function
#user_input = "tell me about the SmartX Pro Phone and the FotoSnap Camera, the DSLR one. Also, what tell me about your TVs?"
user_input = " Tell me what all the products available"
response = process_user_message(user_input, "You are a Service Assistant",False)
print(response)

Here is the list of products currently available in our catalog:

### **1. Computers and Laptops**
*   **TechPro Ultrabook**
*   **BlueWave Gaming Laptop**
*   **PowerLite Convertible**
*   **TechPro Desktop**
*   **BlueWave Chromebook**

### **2. Smartphones and Accessories**
*   **SmartX ProPhone**
*   **MobiTech PowerCase**
*   **SmartX MiniPhone**
*   **MobiTech Wireless Charger**
*   **SmartX EarBuds**

### **3. Televisions and Home Theater Systems**
*   **CineView 4K TV**
*   **SoundMax Home Theater**
*   **CineView 8K TV**
*   **SoundMax Soundbar**
*   **CineView OLED TV**

### **4. Gaming Consoles and Accessories**
*   **GameSphere X**
*   **ProGamer Controller**
*   **GameSphere Y**
*   **ProGamer Racing Wheel**
*   **GameSphere VR Headset**

### **5. Audio Equipment**
*   **AudioPhonic Noise-Canceling Headphones**
*   **WaveSound Bluetooth Speaker**
*   **AudioPhonic True Wireless Earbuds**
*   **WaveSound Soundbar**
*   **AudioPhonic Turntable**

### **6. Cameras and Camcord

In [14]:
def collect_messages(message, debug=False):
    user_input = inp.value
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    context = "You are a Service Assistant"
    
    #response, context = process_user_message(user_input, context, debug=False)
    response = process_user_message(user_input, context, debug=False)
    #context.append({'role': 'assistant', 'content': f"{response}"})
    context = ""
    panels.append(pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(pn.Row('Assistant:', pn.pane.Markdown(response, width=600, css_classes=['assistant-response'])))

    return pn.Column(*panels)

In [15]:
pn.extension(raw_css=['''
.assistant-response {
    background-color: #F6F6F6;
    padding: 10px;
    border-radius: 5px;
}
'''])

In [16]:
panels = []  # collect display
context = "You are a Service Assistant"

inp = pn.widgets.TextInput(placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(label='Service Assistant', name='Service Assistant')
    [2] ParamFunction(function, _pane=Str, defer_load=False, height=300, loading_indicator=True)